In [2]:
%pip install fsspec

from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import torch
import os
import shutil

# Set Hugging Face cache directory to avoid interference
os.environ['HF_HOME'] = 'C:\\Users\\kavin\\Desktop\\hf_cache'

# Step 1: Clear existing cache to avoid permission issues
cache_path = "C:\\Users\\kavin\\.cache\\huggingface\\datasets\\csv"
shutil.rmtree(cache_path, ignore_errors=True)

# Step 2: Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Step 3: Load the dataset without cache
dataset = load_dataset(
    "csv",
    data_files="../Dataset/data1.csv",
    cache_dir="C:\\Users\\kavin\\Desktop\\hf_temp_cache",  # Use a separate temp cache
    load_from_cache_file=False  # Avoid using existing cache
)

# Step 4: Split the dataset into train and validation sets (80% train, 20% test)
split_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Step 5: Load the GPT-2 model and tokenizer
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)  # Move model to selected device
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 tokenizer does not include padding by default
tokenizer.pad_token = tokenizer.eos_token

# Step 6: Preprocess the dataset
def preprocess_function(examples):
    # Combine question, context, and answer into a single string
    inputs = [f"question: {q} context: {c} answer: {a}" for q, c, a in zip(examples['question'], examples['context'], examples['answer'])]
    
    # Tokenize the combined string
    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)  # Adjust max_length as needed
    
    # The labels should be the same as the input IDs for causal language modeling
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# Tokenize datasets with fewer workers to avoid file conflicts
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, num_proc=1)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True, num_proc=1)

# Step 7: Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Reduce batch size for longer sequences
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # Simulate larger batches
    save_steps=10_000,
    save_total_limit=3,  # Limit saved models to save disk space
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="steps",
    eval_steps=5_000,
    load_best_model_at_end=True,
    fp16=True,  # Mixed precision training if supported
    weight_decay=0.01,  # Regularization to avoid overfitting
    logging_first_step=True,
    max_steps=50_000,  # Adjust based on dataset size
    dataloader_num_workers=0,  # Prevent conflicts during data loading
    disable_tqdm=False
)

# Step 8: Initialize the Trainer
trainer = Trainer(
    model=model,                         # The model to be trained
    args=training_args,                  # Training arguments
    train_dataset=tokenized_train_dataset,  # Training dataset
    eval_dataset=tokenized_eval_dataset,    # Evaluation dataset
    tokenizer=tokenizer                  # Tokenizer for text processing
)

# Step 9: Train the model
trainer.train()

# Step 10: Save the fine-tuned model and tokenizer
model.save_pretrained('./../fine_tuned_gpt2_2')
tokenizer.save_pretrained('./../fine_tuned_gpt2_2')

# Step 11: Evaluate the model
results = trainer.evaluate()
print(results)


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.2.0 requires aiohttp, which is not installed.
datasets 3.2.0 requires filelock, which is not installed.
datasets 3.2.0 requires pyyaml>=5.1, which is not installed.
datasets 3.2.0 requires requests>=2.32.2, which is not installed.
datasets 3.2.0 requires tqdm>=4.66.3, which is not installed.
evaluate 0.4.3 requires requests>=2.19.0, which is not installed.
evaluate 0.4.3 requires tqdm>=4.62.1, which is not installed.
huggingface-hub 0.26.2 requires filelock, which is not installed.
huggingface-hub 0.26.2 requires pyyaml>=5.1, which is not installed.
huggingface-hub 0.26.2 requires requests, which is not installed.
huggingface-hub 0.26.2 requires tqdm>=4.42.1, which is not installed.
huggingface-hub 0.26.2 requires typing-extensions>=3.7.4.3, which is not installed.
torch 2.5.1+cu118 requires filelock, w

Error importing huggingface_hub.hf_api: No module named 'requests'


ModuleNotFoundError: No module named 'requests'

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# 1. Load the fine-tuned model and tokenizer
model_name_or_path = "./../fine_tuned_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path)
model = GPT2LMHeadModel.from_pretrained(model_name_or_path)

# Ensure the pad token is set
tokenizer.pad_token = tokenizer.eos_token

# 2. Prepare the input
def prepare_input(question, context):
    """
    Prepare the input string in the same format as used during training.
    """
    input_text = f"question: {question} context: {context} answer:"
    return input_text

# question = "what supervised learninng?"
# context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."
question = "What is communication"
context = "Parallel systems involve multiple processors working on a single task simultaneously, often sharing memory and resources within a single system. An example is a supercomputer used for weather forecasting. Distributed systems, on the other hand, consist of independent systems working together over a network to solve problems, such as cloud platforms like AWS or Google Cloud. While parallel systems focus on speed and performance for a single task, distributed systems prioritize resource sharing and handling separate tasks."

input_text = prepare_input(question, context)

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# 3. Generate the output
output_ids = model.generate(
    input_ids=input_ids,
    max_length=200,  # Adjust based on the expected answer length
    num_return_sequences=1,  # Number of outputs to generate
    temperature=0.7,  # Controls creativity
    top_k=50,  # Filters to top K likely tokens
    top_p=0.95,  # Nucleus sampling
    pad_token_id=tokenizer.eos_token_id,  # Ensure the model uses the correct pad token
)

# Decode the generated output
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 4. Extract the answer
# The answer follows "answer:" in the generated text
answer = output_text.split("answer:")[-1].strip()

# 5. Print the result
print("Generated Answer:", answer)
